In [1]:
import open3d as o3d
import numpy as np

def pick_points(file_path):
    pts = np.load(file_path)
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    
    print("--- Instructions ---")
    print("1. Press 'Shift + Left Click' to select a point.")
    print("2. A small sphere will appear at the selection.")
    print("3. After selecting 3 points, press 'Q' to close the window.")
    
    # Visualize and allow editing
    vis = o3d.visualization.VisualizerWithEditing()
    vis.create_window(window_name=f"Selecting points for: {file_path}")
    vis.add_geometry(pcd)
    vis.run()  # User picks points here
    vis.destroy_window()
    
    # Retrieve the indices of the picked points
    picked_indices = vis.get_picked_points()
    
    # Map indices back to coordinate values
    picked_coords = pts[picked_indices]
    
    return picked_coords


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Execute for plank
plank_pts = pick_points("plank.npy")

print("\nSelected Coordinates (Plank):")
for i, pt in enumerate(plank_pts):
    print(f"Point {i+1}: {pt}")

--- Instructions ---
1. Press 'Shift + Left Click' to select a point.
2. A small sphere will appear at the selection.
3. After selecting 3 points, press 'Q' to close the window.
[Open3D INFO] No point has been picked.
[Open3D INFO] Picked point #1955 (0.025, 0.46, 0.21) to add in queue.
[Open3D INFO] No point has been picked.
[Open3D INFO] No point has been picked.
[Open3D INFO] Picked point #619 (0.0063, -0.033, 0.39) to add in queue.
[Open3D INFO] Picked point #860 (0.032, -0.27, 0.22) to add in queue.

Selected Coordinates (Plank):
Point 1: [0.02458762 0.45881905 0.21354258]
Point 2: [ 0.00626671 -0.03265827  0.39321075]
Point 3: [ 0.03160511 -0.26829858  0.22452223]


In [3]:
# Execute for ref
ref_pts = pick_points("ref.npy")

print("\nSelected Coordinates (Ref):")
for i, pt in enumerate(ref_pts):
    print(f"Point {i+1}: {pt}")

--- Instructions ---
1. Press 'Shift + Left Click' to select a point.
2. A small sphere will appear at the selection.
3. After selecting 3 points, press 'Q' to close the window.
[Open3D INFO] Picked point #1235 (-0.019, 0.47, 0.42) to add in queue.
[Open3D INFO] Picked point #329 (-0.011, 0.051, 0.62) to add in queue.
[Open3D INFO] Picked point #6739 (-0.027, -0.24, 0.47) to add in queue.

Selected Coordinates (Ref):
Point 1: [-0.0191962   0.46576235  0.41858544]
Point 2: [-0.01139952  0.05100758  0.6162759 ]
Point 3: [-0.02712306 -0.24053061  0.47373874]


In [8]:
import numpy as np
import open3d as o3d

# Define the 3 point correspondences
ref_points = np.array([
    [-0.01919620,  0.46576235,  0.41858544],
    [-0.01139952,  0.05100758,  0.61627590],
    [-0.02712306, -0.24053061,  0.47373874]
])

plank_points = np.array([
    [ 0.02458762,  0.45881905,  0.21354258],
    [ 0.00626671, -0.03265827,  0.39321075],
    [ 0.03160511, -0.26829858,  0.22452223]
])

# 1. Center the points
plank_mean = np.mean(plank_points, axis=0)
ref_mean = np.mean(ref_points, axis=0)

plank_centered = plank_points - plank_mean
ref_centered = ref_points - ref_mean

# 2. Calculate variance (sum of squared distances from centroid)
var_plank = np.mean(np.sum(plank_centered**2, axis=1))
var_ref = np.mean(np.sum(ref_centered**2, axis=1))

# 3. Calculate Scale factor (s = sqrt(var_ref / var_plank) if we assume pure scaling)
# Or more accurately, using the covariance-based Umeyama approach:
covariance = (ref_centered.T @ plank_centered) / 3
U, S, Vt = np.linalg.svd(covariance)
scale = (1.0 / var_plank) * np.sum(S)

print(f"Calculated Scaling Factor: {scale:.6f}")

Calculated Scaling Factor: 0.957454


In [9]:
# Load full plank data
full_plank_pts = np.load("plank.npy")

# Create PointCloud objects
pcd_original = o3d.geometry.PointCloud()
pcd_original.points = o3d.utility.Vector3dVector(full_plank_pts)
pcd_original.paint_uniform_color([0.5, 0.5, 0.5]) # Grey

# Create Scaled PointCloud
pcd_scaled = o3d.geometry.PointCloud()
pcd_scaled.points = o3d.utility.Vector3dVector(full_plank_pts * scale)
pcd_scaled.paint_uniform_color([0, 0.7, 0]) # Green

# Translate the scaled version to the side for side-by-side view
# We move it by the bounding box width of the original
bbox = pcd_original.get_axis_aligned_bounding_box()
extent = bbox.get_extent()[0] 
pcd_scaled.translate([extent * 1.5, 0, 0])

print("Visualizing: Original (Grey) vs Scaled (Green)")
o3d.visualization.draw_geometries([pcd_original, pcd_scaled], 
                                  window_name="Side-by-Side Comparison")

Visualizing: Original (Grey) vs Scaled (Green)


In [10]:
import open3d as o3d
import numpy as np

# 1. Load the original reference cloud
ref_pts_full = np.load("ref.npy")
pcd_ref = o3d.geometry.PointCloud()
pcd_ref.points = o3d.utility.Vector3dVector(ref_pts_full)
pcd_ref.paint_uniform_color([1, 0.706, 0]) # Orange for Reference
pcd_ref.translate(-pcd_ref.get_center()) # Center it at origin

# 2. Prepare the Scaled Plank cloud
# (Using the 'scale' variable from your previous Umeyama calculation)
pcd_plank_scaled = o3d.geometry.PointCloud()
pcd_plank_scaled.points = o3d.utility.Vector3dVector(full_plank_pts * scale)
pcd_plank_scaled.paint_uniform_color([0, 0.651, 0.929]) # Blue for Scaled Plank
pcd_plank_scaled.translate(-pcd_plank_scaled.get_center()) # Center it at origin

# 3. Move the Plank to the right so they aren't on top of each other
# We calculate the width (x-axis extent) to know how far to move it
ref_extent = pcd_ref.get_axis_aligned_bounding_box().get_extent()
offset = ref_extent[0] * 1.2 # Move it 1.2x the width of the ref
pcd_plank_scaled.translate([offset, 0, 0])

print("Visualizing Side-by-Side:")
print("LEFT (Orange): Original Reference")
print("RIGHT (Blue): Scaled Plank")

# 4. Visualize
# Note: Press 'R' in the window to reset the camera view if it looks weird
o3d.visualization.draw_geometries([pcd_ref, pcd_plank_scaled], 
                                  window_name="Ref vs Scaled Plank",
                                  width=1000, height=700)

Visualizing Side-by-Side:
LEFT (Orange): Original Reference
RIGHT (Blue): Scaled Plank


In [11]:
import numpy as np
import os

# 1. Load the original plank data if not already in memory
# full_plank_pts = np.load("plank.npy") 

# 2. Apply the scaling factor calculated from your Umeyama step
scaled_plank_pts = full_plank_pts * scale

# 3. Define the output filename
output_filename = "plank_scaled.npy"

# 4. Save to the current directory
np.save(output_filename, scaled_plank_pts)

# Verification
if os.path.exists(output_filename):
    print(f"Successfully saved: {output_filename}")
    print(f"New shape: {scaled_plank_pts.shape}")
    print(f"Data scale verified: Max value in scaled is {np.max(scaled_plank_pts):.4f}")
else:
    print("Error: File was not saved.")

Successfully saved: plank_scaled.npy
New shape: (2000, 3)
Data scale verified: Max value in scaled is 0.8354
